# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

In [11]:
# ==========================================================
# BUILDING A MODULAR DATA SANITIZATION & EXPLORATION ENGINE
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder
)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import chi2_contingency
from IPython.display import HTML


# ==========================================================
# PLOTTING METHODS CLASS
# ==========================================================

class PlottingMethods:

    @staticmethod
    def bar_chart(df, column):
        fig = px.bar(
            df[column].value_counts().reset_index(),
            x='index',
            y=column,
            title=f'Bar Chart of {column}'
        )
        return HTML(fig.to_html())

    @staticmethod
    def pie_chart(df, column):
        fig = px.pie(
            df,
            names=column,
            title=f'Pie Chart of {column}'
        )
        return HTML(fig.to_html())

    @staticmethod
    def histogram(df, column):
        fig = px.histogram(
            df,
            x=column,
            title=f'Histogram of {column}'
        )
        return HTML(fig.to_html())


# ==========================================================
# DATA INSPECTOR CLASS
# ==========================================================

class DataInspector:

    def __init__(self, df=None):
        self.df = df

    # ------------------------------------------------------
    # Upload Data
    # ------------------------------------------------------

    def upload_data(self, path):

        self.df = pd.read_csv(
            path,
            na_values=['?', 'n/a', 'NULL', ' ']
        )

        for col in self.df.columns:

            converted = pd.to_numeric(
                self.df[col],
                errors='coerce'
            )

            if not converted.isna().all():
                self.df[col] = converted.combine_first(
                    self.df[col]
                )

        return self.df

    # ------------------------------------------------------
    # Summary
    # ------------------------------------------------------

    def get_summary(self):

        return {
            "rows": len(self.df),
            "columns": len(self.df.columns),
            "numeric_columns":
                self.df.select_dtypes(
                    include=np.number
                ).columns.tolist(),

            "categorical_columns":
                self.df.select_dtypes(
                    exclude=np.number
                ).columns.tolist(),

            "preview":
                self.df.head(20)
        }

    # ------------------------------------------------------
    # Missing Values
    # ------------------------------------------------------

    def handle_missing_values(
        self,
        strategy="mean",
        constant_value=0
    ):

        for col in self.df.columns:

            if self.df[col].isna().sum() == 0:
                continue

            if (
                strategy == "mean"
                and pd.api.types.is_numeric_dtype(
                    self.df[col]
                )
            ):
                self.df[col] = self.df[col].fillna(
                    self.df[col].mean()
                )

            elif (
                strategy == "median"
                and pd.api.types.is_numeric_dtype(
                    self.df[col]
                )
            ):
                self.df[col] = self.df[col].fillna(
                    self.df[col].median()
                )

            elif strategy == "mode":

                self.df[col] = self.df[col].fillna(
                    self.df[col].mode()[0]
                )

            elif strategy == "constant":

                self.df[col] = self.df[col].fillna(
                    constant_value
                )

    # ------------------------------------------------------
    # Duplicate Removal
    # ------------------------------------------------------

    def remove_duplicates(self):

        self.df = self.df.drop_duplicates()

    # ------------------------------------------------------
    # Outlier Detection
    # ------------------------------------------------------

    def handle_outliers(
        self,
        column,
        delete=False
    ):

        q1 = self.df[column].quantile(0.25)
        q3 = self.df[column].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (
            (self.df[column] < lower)
            |
            (self.df[column] > upper)
        )

        if delete:
            self.df = self.df[~mask]

        return self.df[mask]

    # ------------------------------------------------------
    # Delete Rows / Columns
    # ------------------------------------------------------

    def delete_rows(self, rows):

        self.df = self.df.drop(rows)

    def delete_columns(self, columns):

        self.df = self.df.drop(
            columns=columns
        )

    # ------------------------------------------------------
    # Normalize Numeric Data
    # ------------------------------------------------------

    def extract_normalized_numeric_data(
        self,
        method="minmax"
    ):

        num_df = self.df.select_dtypes(
            include=np.number
        )

        if method == "minmax":
            scaler = MinMaxScaler()

        elif method == "standard":
            scaler = StandardScaler()

        else:
            scaler = RobustScaler()

        normalized = scaler.fit_transform(
            num_df
        )

        return pd.DataFrame(
            normalized,
            columns=num_df.columns
        )

    # ------------------------------------------------------
    # Normalize Categorical Data
    # ------------------------------------------------------

    def extract_normalized_categorical_data(
        self,
        method="onehot"
    ):

        cat_df = self.df.select_dtypes(
            exclude=np.number
        )

        if len(cat_df.columns) == 0:
            return pd.DataFrame()

        if method == "onehot":

            encoder = OneHotEncoder(
                sparse_output=False,
                handle_unknown="ignore"
            )

            transformed = encoder.fit_transform(
                cat_df
            )

            return pd.DataFrame(
                transformed,
                columns=encoder.get_feature_names_out(
                    cat_df.columns
                )
            )

        elif method == "ordinal":

            encoder = OrdinalEncoder()

            transformed = encoder.fit_transform(
                cat_df
            )

            return pd.DataFrame(
                transformed,
                columns=cat_df.columns
            )

        else:

            encoder = OrdinalEncoder()

            temp = pd.DataFrame(
                encoder.fit_transform(cat_df),
                columns=cat_df.columns
            )

            return (
                temp - temp.min()
            ) / (
                temp.max() - temp.min()
            )

    # ------------------------------------------------------
    # Merge Data
    # ------------------------------------------------------

    def merge_normalized_data(self):

        numeric_data = self.extract_normalized_numeric_data()

        categorical_data = (
            self.extract_normalized_categorical_data()
        )

        return pd.concat(
            [numeric_data, categorical_data],
            axis=1
        )

    # ------------------------------------------------------
    # Numeric Distribution
    # ------------------------------------------------------

    def plot_numeric_distribution(
        self,
        column
    ):

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=[
                "Violin Plot",
                "Scatter Plot",
                "Histogram"
            ]
        )

        fig.add_trace(
            go.Violin(
                x=self.df[column],
                box_visible=True
            ),
            row=1,
            col=1
        )

        fig.add_trace(
            go.Scatter(
                y=self.df[column],
                mode="markers"
            ),
            row=1,
            col=2
        )

        fig.add_trace(
            go.Histogram(
                x=self.df[column]
            ),
            row=1,
            col=3
        )

        fig.show()

    # ------------------------------------------------------
    # Relationship Analysis
    # ------------------------------------------------------

    def plot_relationship(
        self,
        x,
        y
    ):

        x_num = pd.api.types.is_numeric_dtype(
            self.df[x]
        )

        y_num = pd.api.types.is_numeric_dtype(
            self.df[y]
        )

        if x_num and y_num:

            fig = px.scatter(
                self.df,
                x=x,
                y=y,
                trendline="ols",
                title=f"{x} vs {y}"
            )

        elif (not x_num) and y_num:

            fig = px.box(
                self.df,
                x=x,
                y=y,
                points="all",
                title=f"{x} vs {y}"
            )

        else:

            temp = pd.crosstab(
                self.df[x],
                self.df[y]
            ).reset_index()

            fig = px.bar(
                temp,
                x=x,
                title=f"{x} vs {y}"
            )

        fig.show()

    # ------------------------------------------------------
    # Correlation Heatmap
    # ------------------------------------------------------

    def plot_all_associations_heatmap(self):

        corr = self.df.select_dtypes(
            include=np.number
        ).corr()

        fig = px.imshow(
            corr,
            text_auto=True,
            title="Correlation Heatmap"
        )

        fig.show()


# ==========================================================
# DEMONSTRATION USING TITANIC DATASET
# ==========================================================

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

inspector = DataInspector()

inspector.df = pd.read_csv(url)

print("\nDATASET SUMMARY")
print(inspector.get_summary())

print("\nMISSING VALUES BEFORE CLEANING")
print(inspector.df.isnull().sum())

inspector.handle_missing_values("median")

print("\nMISSING VALUES AFTER CLEANING")
print(inspector.df.isnull().sum())

print("\nNORMALIZED DATA")
normalized_df = inspector.merge_normalized_data()
print(normalized_df.head())

print("\nOUTLIERS IN FARE")
print(inspector.handle_outliers("Fare").head())

inspector.plot_numeric_distribution("Age")

inspector.plot_relationship(
    "Pclass",
    "Fare"
)

inspector.plot_all_associations_heatmap()


DATASET SUMMARY
{'rows': 891, 'columns': 12, 'numeric_columns': ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], 'categorical_columns': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], 'preview':     PassengerId  Survived  Pclass  \
0             1         0       3   
1             2         1       1   
2             3         1       3   
3             4         1       1   
4             5         0       3   
5             6         0       3   
6             7         0       1   
7             8         0       3   
8             9         1       3   
9            10         1       2   
10           11         1       3   
11           12         1       1   
12           13         0       3   
13           14         0       3   
14           15         0       3   
15           16         1       2   
16           17         0       3   
17           18         1       2   
18           19         0       3   
19           20         1       3   

   